<a href="https://colab.research.google.com/github/gitmystuff/DSChunks/blob/main/Categorical_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Categorical Encoding

## Categorical Encoding
* Sklearn One Hot Encoding
* Dummy Trap
* Pandas get_dummies
* Labelizer
* Weight of Evidence
* Frequency Encoding

### Categorical Data
* Nominal (Cat or Dog)
* Ordinal (Grades)
* Works better for limited labels in a category
* Engineer features with many labels

### Multicollinearity
* Predictors need to be independent of each other
* https://www.theanalysisfactor.com/multicollinearity-explained-visually/
* https://statisticsbyjim.com/regression/multicollinearity-in-regression-analysis/
* Cats_and_Dogs = [Cat, Dog, Dog, Cat, Cat, Dog]
* Cats = [1, 0, 0, 1, 1, 0]
* Dogs = [0, 1, 1, 0, 0, 1]

### Mismatch in Training and Test

* Some labels in the train set don't show up in the test set

https://towardsdatascience.com/beware-of-the-dummy-variable-trap-in-pandas-727e8e6b8bde

In [1]:
# sklearn OneHotEncoder
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
# https://stackoverflow.com/questions/50473381/scikit-learns-labelbinarizer-vs-onehotencoder
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelBinarizer

pets = ['cat', 'dog', 'cat', 'cat', 'dog', 'turtle', 'cat', 'cat', 'turtle', 'dog', 'cat']
le = LabelEncoder()
int_values = le.fit_transform(pets)
print('Pets:', pets)
print('Label Encoder:', int_values)
int_values = int_values.reshape(len(int_values), 1)

ohe = OneHotEncoder(sparse_output=False)
ohe = ohe.fit_transform(int_values)
pets_ohe = zip(pd.Series(pets),ohe)
print('One Hot Encoder')
for pet in pets_ohe:
    print(pet[0], '\t', pet[1])
lb = LabelBinarizer()
print('Label Binarizer:\n', lb.fit_transform(int_values))

Pets: ['cat', 'dog', 'cat', 'cat', 'dog', 'turtle', 'cat', 'cat', 'turtle', 'dog', 'cat']
Label Encoder: [0 1 0 0 1 2 0 0 2 1 0]
One Hot Encoder
cat 	 [1. 0. 0.]
dog 	 [0. 1. 0.]
cat 	 [1. 0. 0.]
cat 	 [1. 0. 0.]
dog 	 [0. 1. 0.]
turtle 	 [0. 0. 1.]
cat 	 [1. 0. 0.]
cat 	 [1. 0. 0.]
turtle 	 [0. 0. 1.]
dog 	 [0. 1. 0.]
cat 	 [1. 0. 0.]
Label Binarizer:
 [[1 0 0]
 [0 1 0]
 [1 0 0]
 [1 0 0]
 [0 1 0]
 [0 0 1]
 [1 0 0]
 [1 0 0]
 [0 0 1]
 [0 1 0]
 [1 0 0]]


Gemini, October 10, 2024

**Label Encoder:**

* **Assigns an integer to each category:**  For example, `['cat', 'dog', 'turtle']` might become `[0, 1, 2]`.
* **Creates an ordinal relationship:** This implies an order where 'dog' (1) is somehow between 'cat' (0) and 'turtle' (2), which is often not meaningful.

**Why it's problematic for features:**

1. **False Order:**  Most machine learning algorithms will interpret these integers as having a meaningful order. This can mislead the model and lead to inaccurate results, especially for algorithms like linear regression or support vector machines.

2. **Distance Distortion:** The numerical distances between the encoded values don't necessarily reflect the true relationships between the categories.  For instance, the distance between 'cat' (0) and 'turtle' (2) is twice the distance between 'cat' (0) and 'dog' (1), which is usually not what we want.

**Why it can be okay for target variables:**

* **Classification tasks:**  Many classification algorithms can handle target variables as integers. The order doesn't usually affect the model's ability to learn the relationship between features and classes.
* **Simplicity:** For binary classification (e.g., 'yes' or 'no'), Label Encoder is a simple way to convert labels to 0 and 1.

**Alternatives for features:**

* **One-Hot Encoding:** Creates separate binary columns for each category, avoiding the ordinality issue. This is generally the preferred method for encoding categorical features.
* **Target Encoding (for some cases):**  Replaces each category with the average target value for that category. Useful when you have many categories and want to avoid high dimensionality from one-hot encoding.

**In summary:**

* **Features:** Avoid Label Encoder unless you're certain the order is meaningful and won't mislead your model.
* **Target Variables:** Can be suitable for classification tasks, especially binary classification.

Always consider the nature of your data and the specific algorithm you're using when choosing an encoding method.


In [2]:
pets = pd.DataFrame(pd.Series(pets), columns=['Pets'])
pets.head()

,Pets
0,cat
1,dog
2,cat
3,cat
4,dog


In [3]:
ohe = OneHotEncoder(sparse_output=False)
ohe_pets = ohe.fit_transform(pets)
pets_df = pd.DataFrame(ohe_pets, columns=ohe.get_feature_names_out(['Pets']))
pets_df

,Pets_cat,Pets_dog,Pets_turtle
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,1.0,0.0
5,0.0,0.0,1.0
6,1.0,0.0,0.0
7,1.0,0.0,0.0
8,0.0,0.0,1.0
9,0.0,1.0,0.0


### Dummy Trap

The Dummy Variable Trap occurs when two or more dummy variables created by one-hot encoding are highly correlated (multi-collinear). This means that one variable can be predicted from the others, making it difficult to interpret predicted coefficient variables in regression models. In other words, the individual effect of the dummy variables on the prediction model can not be interpreted well because of multicollinearity.

https://www.learndatasci.com/glossary/dummy-variable-trap/

In [4]:
# before removing just one variable
pets_df.corr()

,Pets_cat,Pets_dog,Pets_turtle
Pets_cat,1.000000,-0.670820,-0.516398
Pets_dog,-0.670820,1.000000,-0.288675
Pets_turtle,-0.516398,-0.288675,1.000000


In [5]:
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_pets = ohe.fit_transform(pets)
pets_df = pd.DataFrame(ohe_pets, columns=ohe.get_feature_names_out(['Pets']))
pets_df

,Pets_dog,Pets_turtle
0,0.0,0.0
1,1.0,0.0
2,0.0,0.0
3,0.0,0.0
4,1.0,0.0
5,0.0,1.0
6,0.0,0.0
7,0.0,0.0
8,0.0,1.0
9,1.0,0.0


In [6]:
pets_df.corr()

,Pets_dog,Pets_turtle
Pets_dog,1.000000,-0.288675
Pets_turtle,-0.288675,1.000000


Gemini, October 10, 2024

**The Dummy Variable Trap**

* **What it is:** When one-hot encoding, we create new binary (0 or 1) columns for each category.  The trap occurs because these new columns become perfectly correlated. If we know the values for 'dog' and 'turtle', we automatically know the value for 'cat'. This redundancy can cause problems in some machine learning models.

* **Why we drop a column:**  To avoid this, we drop one of the one-hot encoded columns. This removes the redundancy and makes our data suitable for a wider range of models.

* **OneHotEncoder:** We use scikit-learn's `OneHotEncoder` to transform the 'pets' series.
* **Correlation:**  We use `.corr()` to see the relationship between the encoded columns. Before dropping, you'll notice high negative correlations. After dropping, this is resolved.
* **Identifying 'cat':** The `identify_cat` function demonstrates how we can still infer the presence of 'cat' even after dropping the 'pets_cat' column.

**Key Takeaways:**

* Dropping a column in one-hot encoding prevents the dummy variable trap.
* Even with a dropped column, the information about the dropped category is still implicitly present in the data.
* Understanding this helps you use one-hot encoding effectively in your machine learning projects.


## Example 1

In [7]:
# one hot encoding
import seaborn as sns

df = sns.load_dataset('penguins')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [8]:
df.isnull().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [9]:
df.dropna(inplace=True)
df.isnull().sum()

,0
species,0
island,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,0


In [10]:
df['species'].value_counts()

,count
species,
Adelie,146
Gentoo,119
Chinstrap,68


In [11]:
df['island'].value_counts()

,count
island,
Biscoe,163
Dream,123
Torgersen,47


In [12]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male


## Dependent Variable = species

In [13]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(['species'], axis=1),
                                                    df['species'],
                                                    test_size=.2,
                                                    random_state=42)

print(X_train.shape)
print(X_test.shape)

(266, 6)
(67, 6)


## Training a Model

Gemini, October 10, 2024

**1. Evaluating Model Generalization**

* **The Goal:** We want models that can accurately predict outcomes on *new, unseen* data, not just memorize the data they were trained on (overfitting).
* **The Problem:** If we train and evaluate a model on the same data, we won't know how well it truly generalizes. It might seem to perform perfectly, but fail miserably in real-world scenarios with different data.
* **The Solution:** Train-test split simulates this "new data" scenario. We split our data:
    * **Training set:** Used to train the model.
    * **Testing set:**  Held back and used *only* for evaluation. This gives us an unbiased estimate of how the model will perform on unseen data.

**2. Avoiding Overfitting**

* **What is overfitting?** A model that overfits learns the training data too well, capturing even noise and random fluctuations. This leads to poor generalization.
* **How train-test split helps:** By evaluating on a separate test set, we can detect overfitting. If the model performs much better on the training set than the test set, it's a sign of overfitting.

**3. Model Selection and Tuning**

* **Comparing models:** Train-test split lets us compare different algorithms or parameter settings. We can train and evaluate each option on the same split, ensuring a fair comparison.
* **Hyperparameter tuning:** We can use the test set performance to fine-tune the model's hyperparameters (settings that control the learning process).

**Analogy:**

Imagine you're studying for a test. You wouldn't just reread your notes over and over. You'd also want to try practice questions from a different source to see if you truly understand the material. The training set is like your notes, and the test set is like those practice questions – they help you gauge your true understanding (or in this case, your model's ability to generalize).

**In summary:** Train-test split is essential for:

* **Estimating real-world performance.**
* **Preventing overfitting.**
* **Making informed choices about models and their settings.**


## One Hot Encoding For Features

### For features with more than 2 unique labels

In [14]:
# use sklearn one hot encoder
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(categories='auto', drop='first', sparse_output=False, handle_unknown='ignore')

cat_features = ['island']
ohe_train = ohe.fit_transform(X_train[cat_features])
ohe_train = pd.DataFrame(ohe_train, columns=ohe.get_feature_names_out(cat_features))
ohe_train.index = X_train.index
X_train = ohe_train.join(X_train)
X_train.drop(cat_features, axis=1, inplace=True)

ohe_test = ohe.transform(X_test[cat_features])
ohe_test = pd.DataFrame(ohe_test, columns=ohe.get_feature_names_out(cat_features))
ohe_test.index = X_test.index
X_test = ohe_test.join(X_test)
X_test.drop(cat_features, axis=1, inplace=True)

In [15]:
X_train.sample(10)

,island_Dream,island_Torgersen,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
338,0.0,0.0,47.2,13.7,214.0,4925.0,Female
80,0.0,1.0,34.6,17.2,189.0,3200.0,Female
252,0.0,0.0,45.1,14.5,207.0,5050.0,Female
198,1.0,0.0,50.1,17.9,190.0,3400.0,Female
313,0.0,0.0,49.5,16.1,224.0,5650.0,Male
117,0.0,1.0,37.3,20.5,199.0,3775.0,Male
292,0.0,0.0,48.2,15.6,221.0,5100.0,Male
142,1.0,0.0,32.1,15.5,188.0,3050.0,Female
294,0.0,0.0,46.4,15.0,216.0,4700.0,Female
146,1.0,0.0,39.2,18.6,190.0,4250.0,Male


In [16]:
import numpy as np

print(np.sort(df['island'].unique()))

['Biscoe' 'Dream' 'Torgersen']


## Bi Label Mapping for Features

### For features with only two unique labels

In [17]:
X_train['sex'] = X_train['sex'].map({'Female': 0, 'Male': 1})
X_test['sex'] = X_test['sex'].map({'Female': 0, 'Male': 1})

## Map Dependent Variable to Number

or use label encoder

In [18]:
import numpy as np

print(np.sort(y_train.unique()))

['Adelie' 'Chinstrap' 'Gentoo']


In [19]:
y_train = y_train.map({'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2})
y_test = y_test.map({'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2})

In [20]:
X_train.head()

,island_Dream,island_Torgersen,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
230,0.0,0.0,40.9,13.7,214.0,4650.0,0
84,1.0,0.0,37.3,17.8,191.0,3350.0,0
303,0.0,0.0,50.0,15.9,224.0,5350.0,1
22,0.0,0.0,35.9,19.2,189.0,3800.0,0
29,0.0,0.0,40.5,18.9,180.0,3950.0,1


In [21]:
y_train.head()

,species
230,2
84,0
303,2
22,0
29,0


## Example 2

In [22]:
# get data
import pandas as pd

houses = pd.read_csv('https://raw.githubusercontent.com/gitmystuff/Datasets/main/house-prices.csv')
print(houses.shape)
print(houses.head())
print(houses.info())

(1460, 81)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008  

In [23]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    houses.drop('SalePrice', axis=1),
    houses['SalePrice'],
    test_size=0.25,
    random_state=42)

In [24]:
print(X_train.shape)
object_cols = X_train.select_dtypes(include='object').columns

cnt_nunique = 0
for col in object_cols:
  n_nunique = X_train[col].nunique()
  cnt_nunique += n_nunique
  print(f"Unique labels for {col}: {n_nunique}")

print(f"Total number of unique labels: {cnt_nunique}")

(1095, 80)
Unique labels for MSZoning: 5
Unique labels for Street: 2
Unique labels for Alley: 2
Unique labels for LotShape: 4
Unique labels for LandContour: 4
Unique labels for Utilities: 2
Unique labels for LotConfig: 5
Unique labels for LandSlope: 3
Unique labels for Neighborhood: 25
Unique labels for Condition1: 9
Unique labels for Condition2: 6
Unique labels for BldgType: 5
Unique labels for HouseStyle: 8
Unique labels for RoofStyle: 6
Unique labels for RoofMatl: 7
Unique labels for Exterior1st: 15
Unique labels for Exterior2nd: 16
Unique labels for MasVnrType: 3
Unique labels for ExterQual: 4
Unique labels for ExterCond: 5
Unique labels for Foundation: 6
Unique labels for BsmtQual: 4
Unique labels for BsmtCond: 4
Unique labels for BsmtExposure: 4
Unique labels for BsmtFinType1: 6
Unique labels for BsmtFinType2: 6
Unique labels for Heating: 6
Unique labels for HeatingQC: 5
Unique labels for CentralAir: 2
Unique labels for Electrical: 4
Unique labels for KitchenQual: 4
Unique labels

### Get Dummies

Categorical encoding can lead to high dimensionality

In [25]:
# using pandas get_dummies
import pandas as pd

X_dummy = pd.get_dummies(X_train, drop_first=True)
print(X_train.shape)
print(X_dummy.shape)

(1095, 80)
(1095, 241)


### One Hot Encoder

In [26]:
! git clone https://github.com/gitmystuff/preppy.git

from preppy.version import __version__
print(__version__)

fatal: destination path 'preppy' already exists and is not an empty directory.
PrepPy Version: 0.1.0


In [27]:
import preppy.utils as utils

X_train = utils.functions.handle_missing_values(X_train)
X_train = utils.functions.do_OHE(X_train)
print(X_train.shape)

(1095, 130)


In [28]:
# verbose=True
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1095 entries, 1023 to 1126
Columns: 130 entries, Id to MiscFeature_TenC
dtypes: float64(92), int64(38)
memory usage: 1.1 MB


! **IMPORTANT**: What ever you do to X_Train do to X_test

One Hot Encoding vs Get_Dummies

Gemini, October 10, 2024

**1. Purpose and Usage:**

* **`pd.get_dummies`:** Primarily designed for data exploration and analysis. It's quick and convenient for creating dummy variables within pandas DataFrames.
* **`OneHotEncoder`:**  Geared towards machine learning tasks. It offers more flexibility and control, especially when building machine learning pipelines.

**2. Handling Unseen Values:**

* **`pd.get_dummies`:**  If your training and testing datasets have different categories, `get_dummies` might create different sets of columns, leading to inconsistencies.
* **`OneHotEncoder`:**  Handles unseen values more robustly. You can use the `handle_unknown='ignore'` parameter to avoid errors when new categories appear in the test data.

**3.  Data Transformation:**

* **`pd.get_dummies`:** Works directly on pandas Series or DataFrames.
* **`OneHotEncoder`:**  Requires you to reshape your data into a 2D array (e.g., using `.values.reshape(-1, 1)`).

**4.  Fitting and Transforming:**

* **`pd.get_dummies`:**  Doesn't have a separate "fit" step. It transforms the data directly.
* **`OneHotEncoder`:**  Follows the typical scikit-learn pattern:
    * `fit`:  Learns the categories from the training data.
    * `transform`:  Applies the encoding to both training and testing data based on what was learned during the `fit` step. This ensures consistency.

**5.  Column Names:**

* **`pd.get_dummies`:** By default, uses the original column name as a prefix for the dummy variables (e.g., `color_red`, `color_blue`).
* **`OneHotEncoder`:**  Uses the `get_feature_names_out()` method to get the names of the generated columns.

**6. Integration with Scikit-learn:**

* **`OneHotEncoder`:** Seamlessly integrates with other scikit-learn tools like pipelines and `ColumnTransformer`, making it easier to build and manage complex preprocessing workflows.

**In summary:**

| Feature | `pd.get_dummies` | `OneHotEncoder` |
|---|---|---|
| **Main Use** | Data analysis | Machine learning |
| **Unseen Values** | Can be inconsistent | Handles with `handle_unknown` |
| **Data Input** | Series/DataFrame | 2D array |
| **Fitting** | No separate fit | Requires `fit` and `transform` |
| **Scikit-learn** | Not designed for pipelines | Integrates well |


**Which one to choose?**

* For quick data exploration and analysis, `pd.get_dummies` is often more convenient.
* For machine learning tasks, especially those involving pipelines and model deployment, `OneHotEncoder` is generally preferred for its consistency and robustness.


### One Hot Encoding Alternatives

For features with many labels

* https://contrib.scikit-learn.org/category_encoders/index.html
* https://medium.com/analytics-vidhya/stop-one-hot-encoding-your-categorical-variables-bbb0fba89809
* https://medium.com/swlh/stop-one-hot-encoding-your-categorical-features-avoid-curse-of-dimensionality-16743c32cea4
* https://towardsdatascience.com/all-about-categorical-variable-encoding-305f3361fd02 (frequency and mean encoding)

### Encoding Order

* Bilabel Mapping (2 labels)
* Frequency (5+ labels)
* One Hot Encoding (3 - 5 labels)